In [ ]:
from dotenv import load_dotenv

from langchain import hub
from langchain_teddynote import logging
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_openai import ChatOpenAI
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import chain
from langchain_teddynote.messages import stream_response
from langchain_teddynote.callbacks import StreamingCallback
from langchain_core.output_parsers import StrOutputParser

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-summary-stuff")

# Stuff

- 문서 목록을 가져와서 모두 프롬프트에 삽입해서 LLM에 전달하는 방식
- 문서가 작을 때와 호출 한 번으로도 처리할 수 있을 정도로 문서 수가 적을 때 적합 (A4용지 50~100 페이지 정도)
- 구현이 매우 단순하고 사용하기 쉬움

In [ ]:
loader_text = TextLoader("data/news.txt")
docs_text = loader_text.load()

In [ ]:
print(f"총 글자수: {len(docs_text[0].page_content)}")
print("\n========= 앞부분 미리보기 =========\n")
print(docs_text[0].page_content[:500])

In [ ]:
# 한국어로 요약을 작성하라는 문구가 담긴 프롬프트

prompt_stuff = hub.pull("teddynote/summary-stuff-documents-korean")
prompt_stuff.pretty_print()

In [ ]:
llm_1 = ChatOpenAI(
    model_name="gpt-4o-mini", 
    streaming=True, 
    temperature=0, 
    callbacks=[StreamingCallback()]
)

In [ ]:
stuff_chain = create_stuff_documents_chain(llm_1, prompt_stuff)
answer = stuff_chain.invoke({"context": docs_text})